# 7.6. Convolutional Neural Networks (LeNet)
D2L의 Convolutional Neural Networks (LeNet)장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. LeNET이란?

LeNet은 초기 CNN 구조 중 하나로, Yann LeCun이 손글씨 숫자 인식을 위해 개발한 모델이다.

LeNet의 구조는 크게 두 부분이다.

1. Convolution 부분
   - 이미지에서 특징을 추출한다.

2. Fully Connected 부분
   - 추출된 특징을 이용해 최종 클래스를 판단한다.

## 2. MLP와 CNN의 차이

예전에 FashionMNIST에서 MLP를 사용했을 때는

```text
이미지

[batch, 1, 28, 28]

↓ Flatten

[batch, 784]

↓ Linear
```
이미지를 펼쳤다. 그러면 이미지의 공간 구조가 사라진다.

```text
MLP:

28 × 28 이미지
↓
784차원 Vector
↓
Linear

CNN:

28 × 28 이미지
↓
Conv
↓
Feature Map
↓
Pooling
↓
더 복잡한 Feature Map
```

CNN은 이미지 공간 정보를 이용하며 특징을 추출한다.

## 3. LeNet 전체 구조

D2L에서 나온 LeNet은 이렇다.

```text
입력
[batch, 1, 28, 28]
↓
Conv2d
1 → 6
kernel=5
padding=2
↓
Sigmoid
↓
AvgPool
2 × 2
↓
Conv2d
6 → 16
kernel=5
↓
Sigmoid
↓
AvgPool
2 × 2
↓
Flatten
↓
Linear
400 → 120
↓
Sigmoid
↓
Linear
120 → 84
↓
Sigmoid
↓
Linear
84 → 10
```

## 4. 첫 번째 Convolution

입력 이미지는 FashionMNIST이므로 [batch, 1, 28, 28] 이다.

흑백 이미지이므로 Channel은 1개이다.

In [ ]:
nn.Conv2d(
    in_channels=1,
    out_channels=6, # 6개의 커널을 사용해 6개의 Feature Map 생성
    kernel_size=5,
    padding=2
)

Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))

[batch, 1, 28, 28] -> [batch, 6, 28, 28] 이다.

$$
H_{\text{out}}
=
\left\lfloor
\frac{H_{\text{in}} + 2P - K}{S}
\right\rfloor
+ 1
$$

28 x 28은 유지된다.

## 5. 첫 번째 Pooling

결과 [batch, 6, 28, 28] 에 Pooling을 적용한다.

In [3]:
nn.AvgPool2d(
    kernel_size=2,
    stride=2
)

AvgPool2d(kernel_size=2, stride=2, padding=0)

그러면 [batch, 6, 28, 28] -> [batch, 6, 14, 14]가 된다.

LeNet이 만들어진 당시에는 지금 흔히 사용하는 ReLU와 Max Pooling 대신 Sigmoid와 Average Pooling을 사용했다고 한다.

## 6. 두 번째 Convolution

현재는 [batch, 6, 14, 14] 상태이다.

In [4]:
nn.Conv2d(
    in_channels=6,
    out_channels=16,
    kernel_size=5
)

Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))

Padding이 없다.

$$
14 - 5 + 1 = 10
$$

그래서 [batch, 6, 14, 14] -> [batch, 16, 10, 10] 이 된다.

## 7. 두 번째 Conv는 무엇을 배울까?

첫 번째 Conv가 원본 이미지에서 선, 모서리, 방향, 간단한 패턴을 찾았다고 생각해보면 현재 입력은 더 이상 원본이 아니다.

두 번째 Conv는 6개의 Feature Map을 입력으로 받는다.

```text
첫 번째 Conv:

원본 Pixel
→ 선, 모서리 등 단순한 특징

두 번째 Conv:

앞에서 찾은 단순한 특징들의 조합
→ 더 복잡한 패턴

예:

선 + 선
→ 모서리

여러 모서리
→ 특정 형태

여러 형태
→ 물체의 부분
```

Loss를 줄이는 방향으로 역전파하며 kernel weight가 학습된다.

## 8. 두 번째 Pooling

현재는 [batch, 16, 10, 10] 상태이다.

In [5]:
nn.AvgPool2d(
    kernel_size=2,
    stride=2
)

AvgPool2d(kernel_size=2, stride=2, padding=0)

적용하면 [batch, 16, 10, 10] -> [batch, 16, 5, 5]가 된다.

현재 흐름
```text
[batch, 1, 28, 28]

↓ Conv

[batch, 6, 28, 28]

↓ Pool

[batch, 6, 14, 14]

↓ Conv

[batch, 16, 10, 10]

↓ Pool

[batch, 16, 5, 5]
```

깊어질수록 Channel은 올라가고 H,W는 줄어드는 패턴이다.

## 9. 왜 갑자기 Flatten을 하나?

CNN으로 특징 추출이 끝났다. 그런데 `nn.Linear`는 이미지 형태인 [batch, C, H, W]를 받을 수 없다. 그래서 펼친다. (`nn.Flatten()`)

그러면 16 x 5 x 5 = 400이므로

[batch, 16, 5, 5] -> [batch, 400] 이 된다.

Convolution 부분에서는 이미지의 공간 구조를 유지하며 특징을 추출하고 특징 추출이 끝난 후에는 분류를 위해서 Feature Map을 하나의 Vector로 펼쳤다.

## 10. Fully Connected Layer

Flatten하고 이후부터는 전에 배운 MLP와 동일하다.

In [6]:
nn.Linear(400, 120)
nn.Sigmoid()

nn.Linear(120, 84)
nn.Sigmoid()

nn.Linear(84, 10)

Linear(in_features=84, out_features=10, bias=True)

```text
[batch, 400]

↓ Linear

[batch, 120]

↓ Linear

[batch, 84]

↓ Linear

[batch, 10]
```

클래스가 10개이기 때문에 마지막은 10이다.

## 11. PyTorch로 LeNet 구현

D2L Helper를 사용하지 않고 작성했다.

In [7]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            # [B, 1, 28, 28]
            nn.Conv2d(
                in_channels=1,
                out_channels=6,
                kernel_size=5,
                padding=2
            ),

            # [B, 6, 28, 28]
            nn.Sigmoid(),

            nn.AvgPool2d(
                kernel_size=2,
                stride=2
            ),

            # [B, 6, 14, 14]

            nn.Conv2d(
                in_channels=6,
                out_channels=16,
                kernel_size=5
            ),

            # [B, 16, 10, 10]
            nn.Sigmoid(),

            nn.AvgPool2d(
                kernel_size=2,
                stride=2
            ),

            # [B, 16, 5, 5]

            nn.Flatten(),

            # [B, 400]

            nn.Linear(16 * 5 * 5, 120),
            nn.Sigmoid(),

            nn.Linear(120, 84),
            nn.Sigmoid(),

            nn.Linear(84, 10)
        )

    def forward(self, x):
        return self.net(x)

## 12. 각 Layer의 Shape 확인하기

In [8]:
model = LeNet()

X = torch.randn(1, 1, 28, 28)

for layer in model.net:
    X = layer(X)

    print(
        layer.__class__.__name__,
        X.shape
    )

Conv2d torch.Size([1, 6, 28, 28])
Sigmoid torch.Size([1, 6, 28, 28])
AvgPool2d torch.Size([1, 6, 14, 14])
Conv2d torch.Size([1, 16, 10, 10])
Sigmoid torch.Size([1, 16, 10, 10])
AvgPool2d torch.Size([1, 16, 5, 5])
Flatten torch.Size([1, 400])
Linear torch.Size([1, 120])
Sigmoid torch.Size([1, 120])
Linear torch.Size([1, 84])
Sigmoid torch.Size([1, 84])
Linear torch.Size([1, 10])


## 13. LeNet 학습

기존 신경망과 동일하다.

1. 입력 이미지를 모델에 넣는다.
2. 순전파로 prediction을 계산한다.
3. prediction과 정답 label을 비교한다.
4. Loss를 계산한다.
5. backward()로 gradient를 계산한다.
6. optimizer가 Weight를 수정한다.

In [11]:
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

100%|██████████| 26.4M/26.4M [00:07<00:00, 3.68MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 128kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.22MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 2.56MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.9)
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for X, y in train_loader:

        # 1. 이전 gradient 초기화
        optimizer.zero_grad()

        # 2. 순전파
        y_hat = model(X)

        # 3. Loss 계산
        loss = criterion(y_hat, y)

        # 4. 역전파
        loss.backward()

        # 5. Weight 업데이트
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs}, "
        f"Loss: {avg_loss:.4f}"
    )

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X, y in test_loader:

        y_hat = model(X)

        pred = y_hat.argmax(dim=1)

        correct += (pred == y).sum().item()
        total += y.size(0)

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.4f}")

## 14. 전체 흐름

```text
FashionMNIST 이미지:

[Batch, 1, 28, 28]

↓

Conv2d(1 → 6)

[Batch, 6, 28, 28]

여러 종류의 낮은 수준 특징 추출

↓

Average Pooling

[Batch, 6, 14, 14]

특징을 압축

↓

Conv2d(6 → 16)

[Batch, 16, 10, 10]

앞의 특징들을 조합하여
더 복잡한 특징 추출

↓

Average Pooling

[Batch, 16, 5, 5]

특징을 다시 압축

↓

Flatten

[Batch, 400]

↓

Linear

[Batch, 120]

↓

Linear

[Batch, 84]

↓

Linear

[Batch, 10]

↓

10개 클래스에 대한 점수

↓

CrossEntropyLoss

↓

Backward

↓

Conv Kernel과 Linear Weight를 모두 학습
```

## 15. 오늘의 정리

- LeNet은 초기 CNN 구조 중 하나이다.
- LeNet은 Conv 부분과 Fully Connected 부분으로 구성된다.
- Conv Layer는 이미지에서 특징을 추출한다.
- Pooling은 Feature Map의 H, W를 줄이며 특징을 압축한다.
- CNN이 깊어지면서 Channel은 증가하고 H, W는 감소하는 경우가 많다.
- 첫 번째 Conv는 비교적 단순한 특징을 학습한다.
- 뒤쪽 Conv는 앞에서 추출한 특징을 조합하여 더 복잡한 특징을 학습한다.
- Feature Map은 마지막에 Flatten하여 Vector로 변환한다.
- Flatten 이후에는 기존 MLP와 마찬가지로 Linear Layer를 사용한다.
- 마지막 출력 개수는 분류할 클래스 개수와 같다.
- CNN도 순전파 → Loss → 역전파 → Optimizer의 동일한 방식으로 학습한다.
- 역전파를 통해 Conv의 Kernel Weight와 Linear의 Weight가 함께 학습된다.